# The Swallowtail Catastrophe and Optical Caustics
## Quartic Phase Modulation and Bifurcation Sets

This notebook adapts the pseudo-spectral PDE solver to simulate the **Swallowtail Catastrophe**. In René Thom's classification of elementary catastrophes, the swallowtail is the next level of complexity after the cusp. While the cusp features a single beak-like point where two fold lines meet, the swallowtail features a central ridge that splits and intersects itself, creating a multi-layered, highly structured interference pattern.

---

## 1. The Swallowtail Catastrophe

In wave optics and quantum mechanics, the mathematical description of the wave field near a swallowtail caustic is given by the **Swallowtail diffraction integral**. 

Just as the Pearcey beam (Cusp) is generated by a *cubic* phase modulation in the transverse direction, the Swallowtail beam is generated by a **quartic** phase modulation. As the beam propagates, the wavefronts fold in a much more complex topology than the cusp, creating the iconic "swallowtail" bifurcation set.

---

## 2. Physical Setup: The Swallowtail Beam

We initialize a Gaussian wave packet propagating in the $+x$ direction, but this time we apply a **quartic phase modulation** in the transverse $y$ direction.

$$
u(x,y,0) = \exp\left(-\frac{x^2}{2\sigma_x^2} - \frac{y^2}{2\sigma_y^2}\right) \cos(k_x x + \alpha y^4)
$$

The quartic term $\alpha y^4$ acts as a spatially varying "kick" to the transverse momentum that grows much faster at the edges than the cubic term. As the beam propagates, the wavefronts fold over themselves multiple times, naturally evolving into the swallowtail geometry.

---

## 3. The Governing Equation

Because the medium is uniform, the governing equation remains the standard 2D scalar wave equation:

$$
\frac{\partial^2 u}{\partial t^2} = c^2 \nabla^2 u
$$

The principal symbol is simply:

$$
a(\xi, \eta) = c^2 (\xi^2 + \eta^2)
$$

Once again, all the complex catastrophe geometry arises purely from the initial conditions and the linear superposition of the wave's Fourier components!

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Wave speed ──
C_SQUARED = 1.0        # c² (m²/s²)
C = np.sqrt(C_SQUARED)

# ── Swallowtail Beam Parameters ──
K_X = 10.0             # Longitudinal wavenumber (carrier frequency)
# CRITICAL: The quartic coefficient must be much smaller than the cubic one!
# The transverse wavenumber is k_y = 4 * alpha * y^3. 
# If alpha is too large, k_y will exceed the Nyquist frequency at the edges, causing severe aliasing.
ALPHA_QUARTIC = 0.1   # Strength of the quartic phase modulation

# ── Envelope widths ──
SIGMA_X = 2.5          # Longitudinal envelope width
SIGMA_Y = 3.0          # Transverse envelope width

# ── Dissipation ──
GAMMA = 0.0            # No damping; we want to see the pure interference pattern

# ── Grid and Time ──
# High resolution is required to resolve the fine diffraction fringes of the swallowtail
Lx, Ly   = 20.0, 10.0
# Nx, Ny = 32, 32 
# Nx, Ny = 64, 64    
Nx, Ny = 256, 128    
# Nx, Ny = 128, 256    

Lt, Nt   = 10.0, 200
# Lt, Nt   = 20.0, 400
# Lt, Nt   = 30.0, 600
# Lt, Nt   = 40.0, 800
# Lt, Nt   = 50.0, 1000
# Lt, Nt   = 60.0, 1200
n_frames = 300

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
x, y, t = sp.symbols('x y t', real=True)
xi, eta = sp.symbols('xi eta', real=True)
u_func  = sp.Function('u')
u       = u_func(t, x, y)

# Standard isotropic wave symbol
symbol_wave = C_SQUARED * (xi**2 + eta**2)

print("Principal symbol (Uniform Medium):")
print("  a(ξ, η) =", symbol_wave)

## 4. Wave equation

In [ ]:
#
#   ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t
#
gamma    = sp.Symbol('gamma', positive=True)
equation = sp.Eq(
    sp.diff(u, t, 2),
    -psiOp(symbol_wave, u) - gamma * sp.diff(u, t)
)
equation_num = equation.subs({gamma: GAMMA})

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp({symbol_wave}, u) - {GAMMA}·∂u/∂t")

## 5. Initial conditions (The Swallowtail Beam)

In [ ]:
def initial_condition_swallowtail(xx, yy):
    """
    Swallowtail beam: Gaussian envelope with quartic transverse phase.
    """
    # 2D Gaussian envelope
    env = np.exp(-(xx**2) / (2 * SIGMA_X**2) - (yy**2) / (2 * SIGMA_Y**2))
    
    # Phase: longitudinal carrier + quartic transverse modulation
    phase = K_X * xx + ALPHA_QUARTIC * yy**4
    
    return env * np.cos(phase)

def initial_velocity_swallowtail(xx, yy):
    """
    Initial velocity based on WKB approximation for rightward propagation: v = -c * du/dx
    We compute the exact spatial derivative to prevent initial transients.
    """
    env = np.exp(-(xx**2) / (2 * SIGMA_X**2) - (yy**2) / (2 * SIGMA_Y**2))
    phase = K_X * xx + ALPHA_QUARTIC * yy**4
    
    # Derivative of the envelope w.r.t x
    d_env_dx = -(xx / SIGMA_X**2) * env
    
    # Product rule: d/dx [env * cos(phase)]
    du_dx = d_env_dx * np.cos(phase) - env * K_X * np.sin(phase)
    
    return -C * du_dx

## 6. Solver setup

In [ ]:
solver = PDESolver(equation_num)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet', # Absorb waves at boundaries
    initial_condition=initial_condition_swallowtail,
    initial_velocity=initial_velocity_swallowtail,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
# Raise the animation size limit
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay='contour', # Contours are essential to see the complex bifurcation lines!
    mode='surface',    
    physical=True      
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('swallowtail_catastrophe.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to swallowtail_catastrophe.mp4")